# Puncturing a clot in 3-D

A thin needle pushed into a blood clot first dents it. Then, all at once, the clot tears open under the tip and the
needle goes in: the force on the needle drops. How hard you have to push before that happens, and how the tear runs,
matter to anyone who removes clots with a wire, a needle or a small robot.

You will build that experiment with MIME, a Python library for simulating microrobots, and its soft-tissue plugin,
`mime-softtissue`: a soft block that can tear, and a rigid tip pushed into it. You will run a small version live, plot
the force against depth, look at runs computed on bigger computers, and watch the clot tear in 3-D.

The clot's material is not made up: its stiffness and toughness were **calibrated** by this plugin against published
tearing tests on human whole-blood clot (Lohr et al. 2025, *Int. J. Fract.*; their raw data, shared CC0), in a
separate replication.

Run the cells one by one from the top (click a cell, then press Shift+Enter).

**Where this runs, and two settings that must come first.** JAX, the library MIME computes with, reads these before
it starts: float64 numbers (the tearing solve needs them), full precision for matrix products on a GPU, and one GPU
code path switched off (it gave wrong numbers on some GPUs). They change nothing on a CPU.

In [ ]:
import os, sys
in_colab = "google.colab" in sys.modules
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_enable_triton_gemm=false")
os.environ.setdefault("JAX_DEFAULT_MATMUL_PRECISION", "highest")
os.environ.setdefault("JAX_ENABLE_X64", "1")
print("in Colab:", in_colab)

**Install (Colab only).** MIME at the exact commit this notebook was checked with, then the plugin from its **private** repository, with a
GitHub token kept in Colab's secrets under the name `GH_PAT` (the key icon on the left, *Add new secret*, and give this
notebook access). The token goes into the install command only; it is never printed, logged or written to a file. Allow
two or three minutes.

In [ ]:
# MIME's exact commit this notebook was last executed against (it holds the surfaces seam, #340). To update it: run
# the notebook end to end against a newer commit of MIME's release/v0.3.0, then put that commit's full sha here.
MIME_COMMIT = "99614e487c90f091e1fcbfa712aa14aa583634b5"
# The plugin's branch that carries this tutorial (its pull request is a draft; "main" once it is merged).
PLUGIN_REF = "tutorial-puncture-3d"
if in_colab:
    import subprocess
    from google.colab import userdata

    def pip_install(spec, secret=None, *options):
        done = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *options, spec], capture_output=True,
                              text=True)
        if done.returncode:
            message = done.stderr if secret is None else done.stderr.replace(secret, "***")
            raise RuntimeError("pip failed:\n" + message[-2000:])

    pip_install("mime-engine[verify] @ git+https://github.com/Microrobotics-Simulation-Framework/MIME@" + MIME_COMMIT)
    _token = userdata.get("GH_PAT")
    pip_install("git+https://" + _token + "@github.com/Microrobotics-Simulation-Framework/mime-softtissue.git@"
                + PLUGIN_REF, _token, "--no-cache-dir")
    del _token
    print("installed")

**Load the pieces we use.** Every number in MIME carries its unit: `mm` is millimetres, `s` seconds, `deg` degrees.
The plugin's pieces come in as `st` (soft tissue).

In [ ]:
import json
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mime import Body, Experiment, State, viz
from mime.assets import reference, resolve
from mime.dynamics import RigidInertial
from mime.scene import Scene, Z
from mime.things import Part
from mime.units import N, deg, kg, kg_m2, m2, m_per_s, m_per_s2, mm, rad_per_s, s

import mime_softtissue.entries as st
from mime_softtissue.studies import results as published

import jax
print("JAX computes on:", jax.devices())

**The settings.** The numbers you can change later, in "Try it yourself". The tip is a sharp cone, like the point of
a needle: its half-angle is how pointed it is. The grid is how finely the clot is cut into small cells for the
computation: finer is more accurate and slower.

In [ ]:
cone_half_angle = 15.0  # degrees
needle_radius = 0.5     # millimetres: the needle's shaft, behind the cone
push_to = 3.0           # millimetres: how deep the tip goes
grid = 0.125            # millimetres: the size of the clot's cells near the tip

**The clot's material, from a record.** MIME keeps measured and calibrated values in *records*: a named file with its
source, its licence and a check of its bytes, which an experiment names instead of copying numbers. This one is the
whole-blood clot calibrated by the plugin's tearing replication. Each value says where it comes from; one is a
placeholder (the clot's Poisson ratio, which tearing tests in thin sheets cannot measure).

In [ ]:
clot_record = "softtissue.whole_blood_clot"
with open(resolve(reference(clot_record))) as f:
    record = json.load(f)
for name, value in record["values"].items():
    print(f"{name:>18} = {value:<12.6g} {record['labels'][name][:96]}")

**Start the experiment.** An experiment holds the whole world. Here "up" is z and there is no gravity (it is tiny
against the forces of the needle). MIME moves time forward in steps of 0.2 seconds; the needle moves at 0.1 mm a second,
so it goes 0.02 mm deeper each step. Each step the clot is solved at rest under the needle (the push is slow, so the
clot's own inertia does not matter).

In [ ]:
experiment = Experiment("clot_puncture", scene=Scene.new(up=Z, units="m"), gravity=m_per_s2(0.0), dt=s(0.2))
speed = 0.1e-3  # metres a second

**The clot.** A cylinder of clot 10 mm across and 5 mm tall, its bottom held. The plugin's block solves how it
stretches and where it tears (a *phase field*: a damage value from 0, intact, to 1, broken, at every point). With
`dimension="axisymmetric"` it computes one slice and assumes the rest is the same all the way round, which is fast;
`dimension="3d"` computes the real 3-D block, later on. The block must be told which tip shape will press on it.

In [ ]:
clot = experiment.add_body(Body(
    "clot",
    dynamics=st.GelBlock(radius=mm(5.0), height=mm(5.0), material=clot_record, h=mm(grid), fine=mm(1.0),
                         dimension="axisymmetric", tip_shape="softtissue.cone_tip"),
    initial=State(position=mm((0.0, 0.0, 0.0))),
))

**The needle.** A rigid body (it does not bend) whose shape is the plugin's cone tip: pointed at the bottom, its very
apex rounded off by 0.02 mm (a real point is never perfectly sharp), widening to the needle's shaft. It starts with its
point just touching the top of the clot, 5 mm up, and moving down.

In [ ]:
needle = st.ConeTip(radius=mm(needle_radius), tip_radius=mm(0.02), half_angle=deg(cone_half_angle), length=mm(6.0))
tip = experiment.add_body(Body(
    "tip",
    dynamics=RigidInertial(),
    parts={"tip": Part(needle)},
    initial=State(position=mm((0.0, 0.0, 5.0)), orientation=(1.0, 0.0, 0.0, 0.0),
                  velocity=m_per_s((0.0, 0.0, -speed)), angular_velocity=rad_per_s((0.0, 0.0, 0.0))),
))
experiment.declare("tip.mass", kg(1.2e-5), reason="a steel needle of a few millimetres (its mass does not matter here)")
experiment.declare("tip.inertia", kg_m2((1e-11, 1e-11, 1e-12)), reason="the same needle's order")

Now the physics, one piece per cell. If you run one of these cells twice, its piece is added twice and MIME will
refuse to build. To fix it, run the cells again from "Start the experiment" (or use Runtime, then Run all).

**The drive.** A servo pushes the needle down at a steady 0.1 mm a second for the whole run. The run has one step
more than the depth needs (the clot sees where the needle was at the end of the step before), rounded up to a whole
number of 6-step frames for the view.

In [ ]:
steps = 6 * int(np.ceil((push_to / 0.02 + 1) / 6))
experiment.attach(st.Servo(speed=m_per_s(speed), insert_steps=steps), body="tip")

**The contact.** Where the needle's surface meets the clot, the clot is pushed out of the way: a very stiff spring on
every point of the clot that the needle's shape would overlap. No friction and no stickiness here (the clot's own
friction and adhesion are not measured). The spring's stiffness is a numerical choice, 100 times the clot's stiffness
over the cell size.

In [ ]:
young = record["values"]["young_modulus"]
experiment.attach(st.TipContact(friction=0.0, adhesion=0.0, stiffness=100.0 * young / (grid * 1e-3)),
                  tip="tip", solid="clot")

**What to measure.** The force the clot puts on the needle and the needle's depth, every step; and the size of the
tear (the area of the crack).

In [ ]:
force_probe = experiment.observe(st.TipForce(solid="clot"))
crack_probe = experiment.observe(st.CrackSize(solid="clot"))

**Run it.** MIME checks the experiment, builds it into fast code, and pushes the needle just over 3 mm into the clot in
156 small steps, keeping everything at every step (`history=True`) and a 3-D frame every sixth step for the view, in a new folder
each time the cell runs. On
a laptop's processor this took about three minutes (one core, the compile included); Colab's free CPU may take
longer.

In [ ]:
t0 = time.time()
built = experiment.build()
clot_run = Path(tempfile.mkdtemp(prefix="clot_")) / "run"  # a fresh folder each time this cell runs
run = built.run_to(clot_run, s(steps * 0.2), history=True, stage_every=s(1.2))
depth = run.result[force_probe].depth.values.to(mm)
force = run.result[force_probe].force.values.to(N) * 1e3  # millinewtons
crack = run.result[crack_probe].area.values.to(m2) * 1e6  # square millimetres
first_drop = next((k for k in range(1, len(force)) if force[k] < 0.97 * force[:k].max()), None)
print(f"Took {time.time() - t0:.0f} s. At {depth[-1]:.2f} mm deep the force on the needle is {force[-1]:.2f} mN.")
if first_drop is not None:
    print(f"It first dropped (by more than 3 %) at {depth[first_drop]:.2f} mm, from {force[:first_drop].max():.2f} mN.")

**Plot the push.** The force against the needle's depth, and the crack's area. Matplotlib, which Colab has, draws it.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(depth, force, color="#2a78d6", linewidth=2)
ax[0].set_xlabel("depth of the tip (mm)"); ax[0].set_ylabel("force on the tip (mN)")
ax[0].set_title("The push: the needle into the clot")
ax[1].plot(depth, crack, color="#d6612a", linewidth=2)
ax[1].set_xlabel("depth of the tip (mm)"); ax[1].set_ylabel("crack area (mm²)")
ax[1].set_title("The tear")
for a in ax:
    a.grid(color="#e1e0d9")
plt.tight_layout(); plt.show()

**Watch it in 3-D.** The clot coloured by damage (dark: intact, yellow: torn) and the needle going in. The block was
computed as one slice; the view turns that slice three quarters of the way round, so you can see inside. Drag to turn
the view, scroll to zoom, and use the play button or the slider. The view needs an internet connection.

In [ ]:
viz.display(clot_run, caption="The axisymmetric clot, revolved for display, coloured by damage (0 to 1).")

## In real 3-D

A needle's tear is not the same all the way round: a sharp tip opens a flat crack, like a slit, which an axisymmetric
slice cannot hold. The plugin's 3-D block computes a quarter of the real block (the other three quarters are its
mirror images) with a fast solver for the large systems that 3-D brings (a multigrid, low-precision where it is safe,
stopping inner iterations early: about a sixth of the plain solver's cost at the same answer).

Changing to 3-D is one setting, `dimension="3d"`, on the clot. It costs more: minutes to compile on Colab's CPU and then
minutes to hours per run, so the runs below were made on GitHub's computers (`examples/puncture.py`) and are loaded
here. The next cell shows the change, made the way you would make your own: a new experiment **based on** the one above,
which the plugin ships saved, with the clot declared again.

In [ ]:
from mime_softtissue.studies.results import saved

# The experiment above, saved in the plugin; a new experiment based on it, its clot made 3-D (the whole block is
# declared again: MIME replaces the one taken, and says so).
experiment_3d = Experiment.based_on(saved("clot_puncture"), name="clot_puncture_3d")
experiment_3d.add_body(Body(
    "clot",
    dynamics=st.GelBlock(radius=mm(5.0), height=mm(5.0), material=clot_record, h=mm(0.25), fine=mm(1.0),
                         dimension="3d", tip_shape="softtissue.cone_tip"),
    initial=State(position=mm((0.0, 0.0, 0.0))),
))
run_3d_live = False  # True to run its first 0.3 mm: several minutes to compile on Colab's CPU, then minutes more
if run_3d_live:
    run_3d = experiment_3d.build().run_to(Path(tempfile.mkdtemp(prefix="clot3d_")) / "run", s(3.6),
                                          stage_every=s(1.2))
    print(run_3d.result.text())

**The converged runs.** Computed on GitHub's runners (4 virtual CPUs each) with the same experiment (`examples/puncture.py`), and
saved in the plugin as small files.

- **Left: the cone into the clot, axisymmetric, at three cell sizes.** The live run above used the coarsest. Where the
  curves differ, the coarse grid is not converged: that difference is the live run's error.
- **Right: the 3-D blunt punch.** A 1 mm flat punch into a placeholder gel (the case the fast 3-D solver was tuned on),
  run here through MIME's `Experiment` with `dimension="3d"`, against the same solver's earlier runs outside MIME (its
  plain baseline and its fast configuration). They should lie on each other: that is the check that MIME's version is
  the solver that was tested. And the same punch into the calibrated clot, a prediction.

In [ ]:
P = published.puncture()
print(P["about"])
runs = P["runs"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for name, colour in (("cone-clot-h0.125", "#9bbbe0"), ("cone-clot-h0.0625", "#2a78d6"), ("cone-clot-h0.03125", "#0b3a75")):
    if name in runs:
        r = runs[name]
        ax[0].plot(r["depth_mm"], np.array(r["force_N"]) * 1e3, color=colour, label=f"cells {r['h_mm']} mm")
ax[0].plot(depth, force, ":", color="k", label="live, this notebook")
ax[0].set_xlabel("depth of the tip (mm)"); ax[0].set_ylabel("force on the tip (mN)")
ax[0].set_title("15-degree cone into whole-blood clot (axisymmetric)"); ax[0].legend(fontsize=8)
ref = P["reference"]
ax[1].plot(ref["baseline"]["depth_mm"], np.array(ref["baseline"]["force_N"]) * 1e3, "s", mfc="none", color="0.5",
           label="3-D solver outside MIME, plain")
ax[1].plot(ref["all2"]["depth_mm"], np.array(ref["all2"]["force_N"]) * 1e3, "^", mfc="none", color="0.2",
           label="3-D solver outside MIME, fast")
for name, colour in (("p3d-gel-h0.2", "#9bbbe0"), ("p3d-gel-h0.1", "#2a78d6")):
    if name in runs:
        r = runs[name]
        ax[1].plot(r["depth_mm"], np.array(r["force_N"]) * 1e3, color=colour,
                   label=f"through MIME, cells {r['h_mm']} mm")
ax[1].set_xlabel("depth of the punch (mm)"); ax[1].set_ylabel("force on the punch (mN)")
ax[1].set_title("1 mm flat punch into a placeholder gel (3-D)"); ax[1].legend(fontsize=8)
for a in ax:
    a.grid(color="#e1e0d9")
plt.tight_layout(); plt.show()
if "p3d-clot-h0.1" in runs:
    r = runs["p3d-clot-h0.1"]
    k = int(np.argmax(r["force_N"]))
    print(f"The same punch into the clot (3-D, a prediction): {r['force_N'][k] * 1e3:.2f} mN at {r['depth_mm'][k]:.2f} mm "
          f"(the run reached {r['depth_mm'][-1]:.2f} mm).")

**Watch the 3-D block tear.** The quarter block that was computed, coloured by damage. The two flat faces at the back
are its planes of symmetry: the crack inside the block shows on them.

In [ ]:
ANIMATED = "p3d-gel-h0.2"
viz.display(published.puncture_run(ANIMATED),
            caption="The 3-D flat punch into the placeholder gel: the computed quarter of the block, coloured by damage (0 to 1).")

## Viscoelasticity

*A slot: this section arrives later today with the plugin's viscoelastic option (a clot relaxes and is stiffer when
pushed faster): the puncture force against the insertion speed, the push on the needle relaxing after it stops, and
tearing that grows gradually.*

## A published finding to match

Fakhouri, Hutchens and Crosby (2015, *Soft Matter* 11, 4723) pushed **sphere-tipped needles** of radius R from about
0.1 to 5 mm into a soft gel and measured the force at which it punctures, Fc. Fregonese and Bacca (2021, *J. Mech. Phys.
Solids*; arXiv 2412.09809) put those data on one curve: the force made dimensionless, Fc / (mu R²), against
mu R / Gc, where mu is the gel's shear modulus (7.2 kPa) and Gc its toughness (25 J/m²); its theory needs a strongly
stiffening material (an Ogden exponent alpha of 9) to reach the data where needles are small.

The points below are **digitised** from their Fig. 5 (Exp. 1: Fakhouri's gel, with its error bars) and so are their
theory curves; only the numbers are used here, cited. The gel is a record too, `softtissue.puncture_gel`; its
phase-field length scale is a declared placeholder, since the paper states none and it sets the model's strength.

**The model.** The same experiment with a sphere-tipped needle (`st.PunchTip` with `fillet` equal to `radius`), and
MIME's `Study`, which runs one experiment at many settings: here the needle's radius, R = 0.5, 1, 2 and 4 mm, all
compiled once. The gel's toughness is held, as in the experiment (a record's values are not a study's axes). These are
axisymmetric runs: a ring-shaped crack is the only kind the axisymmetric block can form, while the gel's real crack is
flat, so these points are a first look, not the replication. The converged 3-D runs are the replication; they are
being computed and will replace these points.

In [ ]:
F = published.fregonese_fig5()
mu, gc = 7200.0, 25.0
e1 = F["exp1"]
x = np.array([p["x"] for p in e1]); y = np.array([p["y"] for p in e1])
low = np.array([p["y_low"] for p in e1]); high = np.array([p["y_high"] for p in e1])
plt.figure(figsize=(8, 5))
plt.errorbar(x, y, yerr=[y - low, high - y], fmt="o", color="#2a78d6", mfc="none", capsize=3,
             label="Fakhouri et al. 2015, digitised from Fregonese and Bacca's Fig. 5")
for name, style in (("alpha2", "-"), ("alpha9", "-.")):
    c = np.array(F["theory"][name])
    keep = (c[:, 0] > 5e-3) & (c[:, 0] < 5)
    plt.plot(c[keep, 0], c[keep, 1], style, color="0.45", label=f"their theory, Ogden alpha = {name[5:]}")
markers = {"sweep-gel-h0.2": ("s", "#d6612a", "cells 0.2 mm"), "sweep-gel-h0.1": ("D", "#8a2be2", "cells 0.1 mm")}
for name, sweep in P.get("sweeps", {}).items():
    m, colour, label = markers.get(name, ("x", "k", name))
    R = np.array([row["R_mm"] for row in sweep["members"]]) * 1e-3
    Fc = np.array([row["peak_N"] for row in sweep["members"]])
    plt.plot(mu * R / gc, Fc / (mu * R ** 2), m, color=colour, ms=8,
             label=f"this model, axisymmetric, {label} (largest force by 8 mm)")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("mu R / Gc"); plt.ylabel("Fc / (mu R²)")
plt.title("Puncture of a soft gel by sphere-tipped needles")
plt.grid(color="#e1e0d9", which="both"); plt.legend(fontsize=8); plt.show()

## What this tutorial claims

**It claims** that every experiment here builds and runs as written through MIME's `Experiment` and `Study`, and
that the outputs shown are what those runs gave; and that MIME's 3-D block is the fast 3-D solver that was tested on its
own: at a coarse grid the two give the same forces to 1e-12, and a 1 % change in the gel's stiffness is caught at every
step. That is a check of the code, not of the physics.

**The clot's material** is a set of three constants (Yeoh C1 363 ± 18 Pa, C2 434 ± 95 Pa, Gc 0.83 ± 0.17 J/m² at a
length scale of 0.25 mm), fitted on the raw data of Lohr et al. (2025, CC0). Its fit's own claim: at that Gc, the model
gives held-out group 5's peak force within 2 % (inside its band) and groups 1-3's 19 to 24 % low, for in-vitro whole-blood
clot in pure shear at 0.2 mm/s; it does not reproduce the notch dependence with one Gc. It holds in pure shear to a
stretch of about 1.5 at 0.2 mm/s; **puncture with these constants is a prediction.**

**It does not claim:**

- **A predicted puncture force for clot.** The clot's numbers were calibrated in tearing tests on thin sheets, not in
  puncture, and no needle-puncture measurement on clot with public numbers was found to check them against. The
  closest, guidewire push forces of about 15 to 56 mN through clot analogues (Reymond et al. 2024, read from its
  abstract only), is a different tip and test. The clot runs are **predictions**, labelled as such.
- **A converged live run.** The live run's cells (0.125 mm) give forces up to about 20 % away from the finer runs'
  (left plot above).
- **A needle's tear from an axisymmetric run.** It can only form ring-shaped cracks; a sharp tip's flat crack needs the
  3-D block.
- **The replication, yet.** The gel's master curve is the published finding to meet; the axisymmetric points are a
  first look, and the 3-D runs are not in this version of the notebook.

**What the model lacks:**

- **Rate.** Clot is viscoelastic (it relaxes over seconds to a minute) and puncture forces rise with needle speed; the
  model is elastic and quasi-static.
- **Friction and adhesion** between the needle and the clot: set to zero here; the contact law has both, unmeasured.
- **Heterogeneity.** One uniform material; real clots have fibrin-rich and red-cell-rich regions, and fluid (about half
  their volume), which the model leaves out.
- **Two declared numbers.** The clot's Poisson ratio (0.45, a placeholder for the solver), and the phase field's length
  scale (0.25 mm for clot, from the tearing calibration; a placeholder for the gel), which sets the model's strength.

## Try it yourself

Change a number in the settings cell, then run every cell again (Runtime, then Run all).

- **A blunter cone:** `cone_half_angle = 30.0`, and compare its curve with the 15-degree cone's.
- **A finer grid:** `grid = 0.0625`. About four times slower; compare it with the converged curves.
- **A sphere-tipped needle:** in the needle's cell, use
  `needle = st.PunchTip(radius=mm(0.5), fillet=mm(0.5), length=mm(6.0))`, and in the clot's cell
  `tip_shape="softtissue.punch_tip"`.

## Adapting this to your own clot and tips

Start from the saved experiment, as the 3-D cell did, and declare again what is yours: the material (a record's name)
and the tip (its shape and dimensions). MIME replaces each declaration it took, warns you, and lists it in
`plan().replaced`; the new experiment records which one it was based on (its name, its input hash; later its DOI).

- **Your material.** Fit your rheology and compression data with the plugin's fitting harness
  (`python -m mime_softtissue.fit manifest.toml --out results/`, the columns in `docs/fit_spec.md`; it fits a neo-Hookean
  or fibre law), write the values into a record of your own (a JSON file with its licence, named in your plugin's
  `records.json`), and name it as `material`. The cell below uses the plugin's gel record as a stand-in.
- **Your toughness.** Calibrate Gc on one of your geometries and predict the others, with the split (which test
  calibrates, which ones test) written down and committed before any run, as the clot's tearing replication did.
- **Your tips.** A flat punch, a rounded or sphere-tipped needle (`st.PunchTip`), a cone or a needle with a conical point
  (`st.ConeTip`), or a headed tip (`st.HeadedTip`), with your dimensions; the block's `tip_shape` names its shape.
- **The contact's stiffness** is 100 times the material's Young's modulus over the cell size: attach the contact again
  when the material changes (it holds a number, not the record).
- **Not supported yet:** tips from a mesh or CAD file (with MIME's next development release); adhesion and the
  retrieval stroke in 3-D; rotation and drilling (with the plugin's first development release); a clot that varies from
  place to place.

In [ ]:
my = Experiment.based_on(saved("clot_puncture"), name="my_clot")
my_record = "softtissue.puncture_gel"  # your fitted record's name
my.add_body(Body(
    "clot",
    dynamics=st.GelBlock(radius=mm(5.0), height=mm(5.0), material=my_record, h=mm(0.125), fine=mm(1.0),
                         dimension="axisymmetric", tip_shape="softtissue.punch_tip"),
    initial=State(position=mm((0.0, 0.0, 0.0))),
))
my.add_body(Body(
    "tip",
    dynamics=RigidInertial(),
    parts={"tip": Part(st.PunchTip(radius=mm(0.5), fillet=mm(0.5), length=mm(6.0)))},  # a sphere-tipped needle
    initial=State(position=mm((0.0, 0.0, 5.0)), orientation=(1.0, 0.0, 0.0, 0.0),
                  velocity=m_per_s((0.0, 0.0, -speed)), angular_velocity=rad_per_s((0.0, 0.0, 0.0))),
))
with open(resolve(reference(my_record))) as f:
    my_young = json.load(f)["values"]["young_modulus"]
my.attach(st.TipContact(friction=0.0, adhesion=0.0, stiffness=100.0 * my_young / 0.125e-3), tip="tip", solid="clot")
for line in my.plan().replaced:
    print("replaced:", line[:160], "...")
my_folder = Path(tempfile.mkdtemp(prefix="my_clot_")) / "my_clot"
my.save(my_folder)  # runs its first steps, about a minute
print("saved", my_folder)